In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("Imports successful")

Imports successful


In [3]:
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = ROOT / "results"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Processed data folder:", PROCESSED_DIR)
print("Results folder:", RESULTS_DIR)

Project root: c:\Users\adars\OneDrive\Pigskin Vision Project\pigskin-vision
Processed data folder: c:\Users\adars\OneDrive\Pigskin Vision Project\pigskin-vision\data\processed
Results folder: c:\Users\adars\OneDrive\Pigskin Vision Project\pigskin-vision\results


In [4]:
SEASON = 2024

pfr_url = (
    "https://github.com/nflverse/nflverse-data/releases/download/"
    f"pfr_advstats/advstats_week_def_{SEASON}.csv"
)

pfr_2024 = pd.read_csv(pfr_url, low_memory=False)

print("Rows:", len(pfr_2024))
print("Columns:", len(pfr_2024.columns))

pfr_2024.head()

Rows: 7992
Columns: 29


,game_id,pfr_game_id,season,week,game_type,team,opponent,pfr_player_name,pfr_player_id,def_ints,...,def_air_yards_completed,def_yards_after_catch,def_times_blitzed,def_times_hurried,def_times_hitqb,def_sacks,def_pressures,def_tackles_combined,def_missed_tackles,def_missed_tackle_pct
0,2024_01_BAL_KC,202409050kan,2024,1,REG,KC,BAL,Nick Bolton,BoltNi00,0,...,17.0,68.0,3,0,0,0.0,0,7,2,0.222
1,2024_01_BAL_KC,202409050kan,2024,1,REG,KC,BAL,Jaylen Watson,WatsJa02,0,...,22.0,31.0,1,0,0,0.0,0,11,0,0.000
2,2024_01_BAL_KC,202409050kan,2024,1,REG,KC,BAL,Chamarri Conner,ConnCh01,0,...,-12.0,37.0,1,0,0,0.0,0,6,3,0.333
3,2024_01_BAL_KC,202409050kan,2024,1,REG,KC,BAL,Bryan Cook,CookBr02,0,...,53.0,6.0,0,0,0,0.0,0,6,1,0.143
4,2024_01_BAL_KC,202409050kan,2024,1,REG,KC,BAL,Justin Reid,ReidJu00,0,...,6.0,7.0,2,0,0,0.0,0,9,1,0.100


In [5]:
print("Rows:", pfr_2024.shape[0])
print("Columns:", pfr_2024.shape[1])

print("\nUnique games:", pfr_2024["game_id"].nunique())
print("Unique players:", pfr_2024["pfr_player_id"].nunique())
print("Weeks:", sorted(pfr_2024["week"].unique()))

Rows: 7992
Columns: 29

Unique games: 285
Unique players: 954
Weeks: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]


In [6]:
pfr_2024["game_type"].value_counts()

game_type
REG    7632
WC      157
DIV     117
CON      55
SB       31
Name: count, dtype: int64

In [7]:
pfr_regular_2024 = pfr_2024[
    pfr_2024["game_type"] == "REG"
].copy()

print("All rows:", len(pfr_2024))
print("Regular-season rows:", len(pfr_regular_2024))
print("Regular-season games:", pfr_regular_2024["game_id"].nunique())

All rows: 7992
Regular-season rows: 7632
Regular-season games: 272


In [8]:
needed_columns = [
    "season",
    "week",
    "game_type",
    "pfr_player_name",
    "pfr_player_id",
    "def_targets",
    "def_completions_allowed",
    "def_ints",
    "def_receiving_td_allowed",
    "def_yards_allowed",
    "def_passer_rating_allowed",
]

pfr_regular_2024[needed_columns].head()

,season,week,game_type,pfr_player_name,pfr_player_id,def_targets,def_completions_allowed,def_ints,def_receiving_td_allowed,def_yards_allowed,def_passer_rating_allowed
0,2024,1,REG,Nick Bolton,BoltNi00,5,5,0,1.0,85.0,158.3
1,2024,1,REG,Jaylen Watson,WatsJa02,7,5,0,0.0,53.0,93.2
2,2024,1,REG,Chamarri Conner,ConnCh01,8,5,0,0.0,25.0,67.2
3,2024,1,REG,Bryan Cook,CookBr02,4,3,0,0.0,59.0,116.7
4,2024,1,REG,Justin Reid,ReidJu00,3,3,0,0.0,13.0,84.7


In [9]:
pfr_regular_2024[needed_columns].dtypes

season                         int64
week                           int64
game_type                        str
pfr_player_name                  str
pfr_player_id                    str
def_targets                    int64
def_completions_allowed        int64
def_ints                       int64
def_receiving_td_allowed     float64
def_yards_allowed            float64
def_passer_rating_allowed    float64
dtype: object

In [10]:
pfr_regular_2024[needed_columns].isna().sum()

season                          0
week                            0
game_type                       0
pfr_player_name                 0
pfr_player_id                   0
def_targets                     0
def_completions_allowed         0
def_ints                        0
def_receiving_td_allowed     3030
def_yards_allowed            3030
def_passer_rating_allowed    2514
dtype: int64

In [11]:
missing_check = pfr_regular_2024[
    pfr_regular_2024["def_yards_allowed"].isna()
][
    [
        "pfr_player_name",
        "def_targets",
        "def_completions_allowed",
        "def_ints",
        "def_receiving_td_allowed",
        "def_yards_allowed",
        "def_passer_rating_allowed",
    ]
]

missing_check.head(10)

,pfr_player_name,def_targets,def_completions_allowed,def_ints,def_receiving_td_allowed,def_yards_allowed,def_passer_rating_allowed
8,Chris Jones,0,0,0,NaN,NaN,NaN
9,Leo Chenal,0,0,0,NaN,NaN,NaN
10,Tershawn Wharton,0,0,0,NaN,NaN,NaN
11,Michael Danna,0,0,0,NaN,NaN,NaN
12,Derrick Nnadi,0,0,0,NaN,NaN,NaN
13,Joshua Williams,1,0,0,NaN,NaN,39.6
22,David Ojabo,0,0,0,NaN,NaN,NaN
23,Broderick Washington Jr.,0,0,0,NaN,NaN,NaN
24,Nnamdi Madubuike,0,0,0,NaN,NaN,NaN
25,Travis Jones,0,0,0,NaN,NaN,NaN


In [12]:
missing_check["def_targets"].value_counts().sort_index()

def_targets
0    2514
1     398
2      82
3      24
4       6
5       5
6       1
Name: count, dtype: int64